# 02 · Auditoría: los dos David López en el TFM

En la tabla de métricas del TFM, los eventos de Opta se sumaron por nombre de jugador. Los dos
David López (Mallorca y Girona) recibieron la suma de los eventos de ambos.

Este cuaderno mide cuánto cambia eso en las cifras del TFM. **No modifica el TFM**: reproduce sus
cálculos sobre la tabla del TFM (copiada en `data/origen/`) y los repite corrigiendo solo los dos
David López con los eventos cruzados por identificador de WhoScored. El resto de jugadores queda
exactamente igual que en el TFM, para aislar el efecto de este fallo.

In [1]:
# Instalamos las librerías del cuaderno; si ya están instaladas no cambia nada
%pip install -q pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importamos os para situarnos en la raíz del proyecto
import os
# Importamos pandas para trabajar con tablas
import pandas as pd

# Si el cuaderno se ejecuta desde notebooks/, subimos a la raíz para usar rutas relativas
if os.path.basename(os.getcwd()) == "notebooks":
    # Cambiamos a la carpeta superior
    os.chdir("..")

# Ensanchamos la salida para que las tablas no se partan
pd.set_option("display.width", 250)

# Tabla de métricas del TFM, tal y como se entregó
tfm = pd.read_csv("data/origen/metricas_jugadores_laliga_2526.csv")
# Mapa de nombres de este proyecto
mapa = pd.read_csv("data/mapa_nombres.csv", dtype={"id_whoscored": "Int64"})
# Acciones de Opta por identificador
eventos = pd.read_csv("data/eventos_por_jugador.csv", dtype={"id_whoscored": "Int64"})
# Altura defensiva, para el grupo de bloque bajo del TFM
altura = pd.read_csv("data/origen/altura_defensiva_laliga_2526.csv")

# Columnas del TFM y su nombre en este proyecto
ACCIONES = {"Tackle": "entradas", "Interception": "intercepciones", "Clearance": "despejes",
            "BlockedPass": "pases_bloqueados", "Aerial": "duelos_aereos"}

## 1. Métricas de cada David López: TFM frente a cruce por identificador

In [3]:
# Filas de los dos David López en el mapa, con sus eventos por identificador
dl = mapa[mapa["nombre_posiciones"] == "David López"].merge(eventos, on="id_whoscored")
# Filas de los dos en la tabla del TFM
dl_tfm = tfm[tfm["jugador"] == "David López"].set_index("equipo")

# Recorremos los dos jugadores
filas = []
for _, f in dl.iterrows():
    # Recorremos las cinco acciones defensivas
    for col_tfm, col in ACCIONES.items():
        # Guardamos el total y el valor por 90 del TFM y del cruce por identificador
        filas.append({"equipo": f["equipo"], "minutos": f["minutos_us"], "id_whoscored": f["id_whoscored"],
                      "accion": col, "total_tfm": dl_tfm.loc[f["equipo"], col_tfm], "total_correcto": f[col],
                      "por90_tfm": dl_tfm.loc[f["equipo"], col_tfm + "_90"],
                      "por90_correcto": f[col] / f["minutos_us"] * 90})
# Pasamos a tabla
comparacion_dl = pd.DataFrame(filas)
# Mostramos con 4 decimales
print(comparacion_dl.round(4).to_string(index=False))

  equipo  minutos  id_whoscored           accion  total_tfm  total_correcto  por90_tfm  por90_correcto
  Girona      300         92691         entradas       15.0               4     4.5000          1.2000
  Girona      300         92691   intercepciones        8.0               0     2.4000          0.0000
  Girona      300         92691         despejes       92.0              11    27.6000          3.3000
  Girona      300         92691 pases_bloqueados        8.0               1     2.4000          0.3000
  Girona      300         92691    duelos_aereos       52.0              10    15.6000          3.0000
Mallorca      827        502424         entradas       15.0              11     1.6324          1.1971
Mallorca      827        502424   intercepciones        8.0               8     0.8706          0.8706
Mallorca      827        502424         despejes       92.0              81    10.0121          8.8150
Mallorca      827        502424 pases_bloqueados        8.0              

## 2. Los 65 centrales del TFM: antes y después

Mismo filtro que el cuaderno 04 del TFM: posición central, 600 minutos o más y las cuatro métricas
defensivas completas. Percentil con `rank(pct=True) * 100`, como en el TFM.

In [4]:
# Métricas que exige el filtro del TFM
DEF = ["Interception_90", "Tackle_90", "Clearance_90", "Aerial_90"]
# Grupo de referencia del TFM
antes = tfm[(tfm["posicion"] == "central") & (tfm["minutes"] >= 600)].dropna(subset=DEF).copy()
# Mostramos su tamaño y si están los David López
print("centrales de referencia:", len(antes))
print("David López en el grupo:", antes.loc[antes["jugador"] == "David López", ["equipo", "minutes"]].to_string(index=False))

# Copia corregida: solo cambian los David López que estén en el grupo
despues = antes.copy()
# Recorremos las filas corregidas
for _, f in dl.iterrows():
    # Fila del jugador en el grupo, si está
    sel = (despues["jugador"] == "David López") & (despues["equipo"] == f["equipo"])
    # Sustituimos cada acción por su valor correcto por 90
    for col_tfm, col in ACCIONES.items():
        # Valor correcto por 90 minutos
        despues.loc[sel, col_tfm + "_90"] = f[col] / f["minutos_us"] * 90

centrales de referencia: 65
David López en el grupo:   equipo  minutes
Mallorca      827


In [5]:
# Medianas antes y después
print("MEDIANAS DE LOS", len(antes), "CENTRALES (por 90 minutos)")
for col in ["Interception_90", "Tackle_90", "Clearance_90", "BlockedPass_90", "Aerial_90"]:
    # Mediana antes, después y diferencia
    a, d = antes[col].median(), despues[col].median()
    print(f"  {col:<16} antes {a:.6f}   después {d:.6f}   diferencia {d - a:+.6f}")

MEDIANAS DE LOS 65 CENTRALES (por 90 minutos)
  Interception_90  antes 1.058824   después 1.058824   diferencia +0.000000
  Tackle_90        antes 1.584118   después 1.559192   diferencia -0.024926
  Clearance_90     antes 5.775401   después 5.775401   diferencia +0.000000
  BlockedPass_90   antes 0.467290   después 0.467290   diferencia +0.000000
  Aerial_90        antes 3.358946   después 3.358946   diferencia +0.000000


## 3. Percentiles antes y después

Fórmula de las cifras publicadas en el TFM (Valjent p9, Llorente p98, bloque bajo 10/19/38): porcentaje del
grupo de referencia con un valor estrictamente menor.

In [6]:
# Jugadores que cita el TFM
CITADOS = [("Diego Llorente", None), ("Martin Valjent", "Mallorca"), ("Antonio Raíllo", "Mallorca"),
           ("David López", "Mallorca")]


# Percentiles de intercepciones y entradas de los citados dentro de un grupo
def percentiles(grupo):
    # Copiamos para no tocar el original
    g = grupo.copy()
    # Recorremos las dos métricas
    for col, corto in [("Interception_90", "int"), ("Tackle_90", "tkl")]:
        # Fórmula de las cifras publicadas en el TFM: % del grupo estrictamente por debajo
        g["p_" + corto] = [(g[col] < v).mean() * 100 for v in g[col]]
    # Filas de los citados
    sel = pd.Series(False, index=g.index)
    for nombre, equipo in CITADOS:
        # Marcamos al jugador por nombre y, si hace falta, por equipo
        sel |= (g["jugador"] == nombre) & ((g["equipo"] == equipo) if equipo else True)
    # Devolvemos las columnas de interés
    return g.loc[sel, ["jugador", "equipo", "Interception_90", "p_int", "Tackle_90", "p_tkl"]].set_index(["jugador", "equipo"])


# Bloque bajo del TFM: los 6 equipos con la línea defensiva más baja
BLOQUE_BAJO = altura.sort_values("altura_defensiva")["equipo"].head(6).tolist()
# Recorremos los dos grupos de referencia del TFM
for titulo, filtro in [(f"LOS {len(antes)} CENTRALES DE LA LIGA", None), ("BLOQUE BAJO " + str(BLOQUE_BAJO), BLOQUE_BAJO)]:
    # Grupo antes y después
    a = antes if filtro is None else antes[antes["equipo"].isin(filtro)]
    d = despues if filtro is None else despues[despues["equipo"].isin(filtro)]
    # Juntamos los percentiles de las dos versiones
    t = percentiles(a).join(percentiles(d), lsuffix="_antes", rsuffix="_despues")
    # Mostramos la tabla
    print(f"\n{titulo} · {len(a)} centrales")
    print(t.round(4).to_string())


LOS 65 CENTRALES DE LA LIGA · 65 centrales


                           Interception_90_antes  p_int_antes  Tackle_90_antes  p_tkl_antes  Interception_90_despues  p_int_despues  Tackle_90_despues  p_tkl_despues
jugador        equipo                                                                                                                                                
Antonio Raíllo Mallorca                   0.6960      12.3077           1.0875      13.8462                   0.6960        12.3077             1.0875        13.8462
Martin Valjent Mallorca                   0.6252       9.2308           1.3591      35.3846                   0.6252         9.2308             1.3591        36.9231
David López    Mallorca                   0.8706      30.7692           1.6324      50.7692                   0.8706        30.7692             1.1971        21.5385
Diego Llorente Real Betis                 2.0639      98.4615           1.9902      75.3846                   2.0639        98.4615             1.9902        75.3846

BLO

## 4. ¿Cambia algún jugador de lado respecto a las medianas de la criba?

La criba del TFM exige intercepciones y entradas por 90 iguales o superiores a la mediana de los 65.

In [7]:
# Recorremos las dos métricas de la criba
for col in ["Interception_90", "Tackle_90"]:
    # Mediana antes y después
    m_a, m_d = antes[col].median(), despues[col].median()
    # Jugadores que cumplen el umbral antes y después
    cumple_a = set(antes.loc[antes[col] >= m_a, "jugador"] + " (" + antes.loc[antes[col] >= m_a, "equipo"] + ")")
    cumple_d = set(despues.loc[despues[col] >= m_d, "jugador"] + " (" + despues.loc[despues[col] >= m_d, "equipo"] + ")")
    # Mostramos quién entra y quién sale
    print(f"{col}: pasan a cumplir {sorted(cumple_d - cumple_a)} · dejan de cumplir {sorted(cumple_a - cumple_d)}")

Interception_90: pasan a cumplir [] · dejan de cumplir []
Tackle_90: pasan a cumplir ['Pedro Bigas (Elche)'] · dejan de cumplir ['David López (Mallorca)']


## 5. Criba de finalistas del TFM, antes y después

Fondo: centrales con 400 a 1.800 minutos y las cuatro métricas completas; finalista quien iguala o supera
las dos medianas de los 65. Se repite con la mediana de entradas corregida y David López corregido.

In [8]:
# Centrales con las cuatro métricas y entre 400 y 1.800 minutos (fondo del TFM)
fondo = tfm[(tfm["posicion"] == "central") & tfm["minutes"].between(400, 1800)].dropna(subset=DEF).copy()
# Versión corregida del fondo: solo cambian los David López
fondo_c = fondo.copy()
# Recorremos los dos David López
for _, f in dl.iterrows():
    # Fila del jugador en el fondo, si está
    sel = (fondo_c["jugador"] == "David López") & (fondo_c["equipo"] == f["equipo"])
    # Sustituimos sus acciones por el valor correcto
    for col_tfm, col in ACCIONES.items():
        # Valor correcto por 90 minutos
        fondo_c.loc[sel, col_tfm + "_90"] = f[col] / f["minutos_us"] * 90


# Finalistas de un fondo con las medianas de un grupo de referencia
def criba(fondo, ref):
    # Umbrales: medianas de intercepciones y entradas
    u_int, u_tkl = ref["Interception_90"].median(), ref["Tackle_90"].median()
    # Jugadores que igualan o superan las dos
    ok = fondo[(fondo["Interception_90"] >= u_int) & (fondo["Tackle_90"] >= u_tkl)]
    # Devolvemos el conjunto de nombres
    return set(ok["jugador"] + " (" + ok["equipo"] + ")")


# Finalistas antes y después
fin_a, fin_d = criba(fondo, antes), criba(fondo_c, despues)
# Mostramos el resultado
print("fondo:", len(fondo), "centrales")
print("finalistas antes:", len(fin_a), sorted(fin_a))
print("finalistas después:", len(fin_d), sorted(fin_d))
print("entran:", sorted(fin_d - fin_a), "· salen:", sorted(fin_a - fin_d))
# Datos de Pedro Bigas, que cruza la mediana de entradas
print(tfm[tfm["jugador"] == "Pedro Bigas"][["jugador", "equipo", "minutes", "Interception_90", "Tackle_90"]].to_string(index=False))

fondo: 33 centrales
finalistas antes: 8 ['Alejandro Francés (Girona)', 'Andrés Castrín (Sevilla)', 'David Carmo (Real Oviedo)', 'Diego Llorente (Real Betis)', 'Eray Cömert (Valencia)', 'Luiz Felipe (Rayo Vallecano)', 'Unai Núñez (Valencia)', 'Zaid Romero (Getafe)']
finalistas después: 8 ['Alejandro Francés (Girona)', 'Andrés Castrín (Sevilla)', 'David Carmo (Real Oviedo)', 'Diego Llorente (Real Betis)', 'Eray Cömert (Valencia)', 'Luiz Felipe (Rayo Vallecano)', 'Unai Núñez (Valencia)', 'Zaid Romero (Getafe)']
entran: [] · salen: []
    jugador equipo  minutes  Interception_90  Tackle_90
Pedro Bigas  Elche     2078         1.256015   1.559192


## 6. Modelos del cuaderno 04 del TFM, antes y después

Se reproduce el código del cuaderno 04 del TFM (celdas 2, 9 y 12) sin cambiar nada: mismas variables,
misma partición por equipo, mismos hiperparámetros y mismas semillas. La única diferencia entre las dos
ejecuciones es la tabla de métricas: la del TFM y la misma con los dos David López corregidos.

Primero se comprueba que la versión "antes" da exactamente las cifras publicadas en el TFM. Si no las
diera, la comparación no mediría nada.

In [9]:
# Importamos numpy para los recuentos de clases
import numpy as np
# Árbol de decisión, como en el TFM
from sklearn.tree import DecisionTreeClassifier
# Bosque aleatorio, como en el TFM
from sklearn.ensemble import RandomForestClassifier
# Validación cruzada con partición por equipo, como en el TFM
from sklearn.model_selection import cross_val_predict, StratifiedGroupKFold
# Métricas de acierto y de calidad de la agrupación
from sklearn.metrics import accuracy_score, confusion_matrix, silhouette_score
# Estandarización previa a k-medias
from sklearn.preprocessing import StandardScaler
# k-medias
from sklearn.cluster import KMeans

# Disparos de la temporada, copiados del TFM
disparos = pd.read_csv("data/origen/disparos_laliga_2526_limpio.csv")
# Equipo y rival de cada partido, copiados del TFM
equipos_partido = pd.read_csv("data/origen/equipos_largo_laliga_2526.csv")

# Tabla del TFM con los dos David López corregidos (el resto, idéntico)
tfm_c = tfm.copy()
# Recorremos los dos David López
for _, f in dl.iterrows():
    # Fila del jugador por nombre y equipo
    sel = (tfm_c["jugador"] == "David López") & (tfm_c["equipo"] == f["equipo"])
    # Sustituimos cada acción por su valor correcto por 90
    for col_tfm, col in ACCIONES.items():
        # Valor correcto por 90 minutos
        tfm_c.loc[sel, col_tfm + "_90"] = f[col] / f["minutos_us"] * 90

# Equipo y rival de cada partido
llave = equipos_partido[["game", "equipo", "rival"]].drop_duplicates()
# Añadimos el rival a cada disparo y descartamos los que no lo tienen
d2 = disparos.merge(llave, left_on=["game", "team"], right_on=["game", "equipo"], how="left").dropna(subset=["rival"])
# Línea del área grande en coordenadas de Understat
X_AREA = 1 - 16.5 / 105
# Disparos en juego abierto desde el centro del área
centro_area = d2[(d2["situation"] == "Open Play") & (d2["location_x"] >= X_AREA) & (d2["location_y"].between(1/3, 2/3))]
# xG concedida por cada equipo
concedido = centro_area.groupby("rival")["xg"].sum()
# Etiqueta del TFM: 1 si el equipo concede menos que la mediana de la liga
etiqueta_equipo = (concedido < concedido.median()).astype(int)

# Variables de los modelos del TFM
VARIABLES = ["Interception_90", "Tackle_90", "Clearance_90", "Aerial_90"]
# Nombres legibles de las variables
NOMBRES = ["Intercepciones", "Entradas", "Despejes", "Duelos aéreos"]
# Candidatos que puntúa el TFM
CANDIDATOS = ["Diego Llorente", "Zaid Romero", "Luiz Felipe", "Andrés Castrín",
              "Eray Cömert", "Alejandro Francés", "Unai Núñez", "David Carmo"]
# Centrales del Mallorca que puntúa el TFM
MALLORCA = ["Martin Valjent", "Antonio Raíllo", "David López", "Marash Kumbulla"]


# Reproduce la celda 2 del cuaderno 04 sobre una tabla de métricas
def supervisado(m):
    # Centrales con 600 minutos y las cuatro métricas
    base = m[(m["posicion"] == "central") & (m["minutes"] >= 600)].dropna(subset=VARIABLES).copy()
    # Etiqueta de su equipo
    base["y"] = base["equipo"].map(etiqueta_equipo)
    # Descartamos los que no tengan etiqueta
    base = base.dropna(subset=["y"])
    # La etiqueta pasa a entero
    base["y"] = base["y"].astype(int)
    # Matriz de variables, etiqueta y grupos de la partición
    X, y, grupos = base[VARIABLES].values, base["y"].values, base["equipo"].values
    # Partición por equipo, como en el TFM
    particion = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    # Los dos modelos del TFM con sus hiperparámetros
    modelos = {"arbol": DecisionTreeClassifier(max_depth=1, random_state=42),
               "bosque": RandomForestClassifier(n_estimators=300, max_depth=4, min_samples_leaf=3, random_state=42)}
    # Tamaño de la muestra, reparto de clases y acierto del clasificador trivial
    r = {"n": len(base), "clases": np.bincount(y).tolist(), "trivial": max(np.bincount(y)) / len(y)}
    # Validación cruzada de cada modelo
    for nombre, modelo in modelos.items():
        # Predicción fuera de muestra
        pred = cross_val_predict(modelo, X, y, cv=particion, groups=grupos)
        # Acierto
        r["acierto_" + nombre] = accuracy_score(y, pred)
        # Matriz de confusión
        r["matriz_" + nombre] = confusion_matrix(y, pred).tolist()
    # Árbol entrenado con todos los datos
    arbol = DecisionTreeClassifier(max_depth=1, random_state=42).fit(X, y)
    # Variable y umbral que aprende
    r["umbral_arbol"] = f"{NOMBRES[arbol.tree_.feature[0]]} <= {arbol.tree_.threshold[0]:.6f}"
    # Bosque entrenado con todos los datos
    bosque = RandomForestClassifier(n_estimators=300, max_depth=4, min_samples_leaf=3, random_state=42).fit(X, y)
    # Importancia de cada variable
    r["importancias"] = dict(zip(NOMBRES, bosque.feature_importances_.round(6)))
    # Todos los centrales con las cuatro métricas, como la ficha del TFM
    ficha = m[m["posicion"] == "central"].dropna(subset=VARIABLES).copy()
    # Probabilidad que les da el bosque
    ficha["prob"] = bosque.predict_proba(ficha[VARIABLES].values)[:, 1]
    # Nos quedamos con los nombres citados en el TFM
    r["prob"] = ficha[ficha["jugador"].isin(CANDIDATOS + MALLORCA)].set_index(["jugador", "equipo"])["prob"]
    # Devolvemos el resultado
    return r


# Ejecutamos con la tabla del TFM y con la corregida
sup_a, sup_d = supervisado(tfm), supervisado(tfm_c)

# Comprobamos que la versión "antes" reproduce las cifras publicadas en el TFM
assert sup_a["n"] == 65 and sup_a["clases"] == [35, 30]
# Aciertos publicados
assert round(sup_a["acierto_arbol"], 3) == 0.523 and round(sup_a["acierto_bosque"], 3) == 0.492
# Clasificador trivial publicado
assert round(sup_a["trivial"], 3) == 0.538
# Matrices de confusión publicadas
assert sup_a["matriz_arbol"] == [[6, 29], [2, 28]] and sup_a["matriz_bosque"] == [[19, 16], [17, 13]]
# Informamos
print("reproducción del TFM: correcta (aciertos, matrices de confusión y reparto de clases)")

reproducción del TFM: correcta (aciertos, matrices de confusión y reparto de clases)


In [10]:
# Comparamos cifra a cifra
for clave in ["n", "clases", "trivial", "acierto_arbol", "matriz_arbol", "acierto_bosque", "matriz_bosque",
              "umbral_arbol", "importancias"]:
    # Valores antes y después
    a, d = sup_a[clave], sup_d[clave]
    # Formato con 6 decimales si es un número
    fmt = (lambda v: f"{v:.6f}") if isinstance(a, float) else str
    # Mostramos la comparación
    print(f"{clave:<15} antes {fmt(a):<75} después {fmt(d)}   {'IGUAL' if a == d else 'CAMBIA'}")

n               antes 65                                                                          después 65   IGUAL
clases          antes [35, 30]                                                                    después [35, 30]   IGUAL
trivial         antes 0.538462                                                                    después 0.538462   IGUAL
acierto_arbol   antes 0.523077                                                                    después 0.523077   IGUAL
matriz_arbol    antes [[6, 29], [2, 28]]                                                          después [[6, 29], [2, 28]]   IGUAL
acierto_bosque  antes 0.492308                                                                    después 0.492308   IGUAL
matriz_bosque   antes [[19, 16], [17, 13]]                                                        después [[20, 15], [18, 12]]   CAMBIA
umbral_arbol    antes Duelos aéreos <= 4.163613                                                   después Duelos aéreos <=

In [11]:
# Probabilidad del bosque para candidatos y centrales del Mallorca
prob = pd.DataFrame({"antes": sup_a["prob"], "despues": sup_d["prob"]})
# Diferencia
prob["diferencia"] = prob["despues"] - prob["antes"]
# Mostramos ordenado por la probabilidad de antes
print(prob.sort_values("antes", ascending=False).round(6).to_string())

                                     antes   despues  diferencia
jugador           equipo                                        
Eray Cömert       Valencia        0.721948  0.722772    0.000824
Zaid Romero       Getafe          0.635031  0.630378   -0.004653
Unai Núñez        Valencia        0.607230  0.612044    0.004814
Diego Llorente    Real Betis      0.566728  0.569444    0.002716
Andrés Castrín    Sevilla         0.538837  0.539816    0.000979
Martin Valjent    Mallorca        0.506693  0.496310   -0.010383
Alejandro Francés Girona          0.411438  0.406170   -0.005269
Marash Kumbulla   Mallorca        0.358995  0.360622    0.001627
Luiz Felipe       Rayo Vallecano  0.331912  0.334902    0.002990
David López       Girona          0.286400  0.358746    0.072346
David Carmo       Real Oviedo     0.268704  0.298033    0.029329
David López       Mallorca        0.257971  0.190188   -0.067783
Antonio Raíllo    Mallorca        0.103059  0.099280   -0.003779


## 7. Agrupación k-medias (celdas 9 y 12 del cuaderno 04), antes y después

k-medias numera los grupos de forma arbitraria. Para comparar, cada grupo de "después" se empareja con
el grupo de "antes" con el que comparte más jugadores, y se comprueba que el emparejamiento es uno a uno.

In [12]:
# Reproduce las celdas 9 y 12 sobre una tabla de métricas
def agrupacion(m):
    # Centrales con 600 minutos y las cuatro métricas
    base = m[(m["posicion"] == "central") & (m["minutes"] >= 600)].dropna(subset=VARIABLES).copy()
    # Estandarización
    escala = StandardScaler()
    # Variables estandarizadas
    X = escala.fit_transform(base[VARIABLES].values)
    # Silueta de k = 2 a 5
    r = {"silueta": {k: silhouette_score(X, KMeans(n_clusters=k, n_init=25, random_state=42).fit(X).labels_)
                     for k in range(2, 6)}}
    # Fondo de rotación: centrales con 400 a 1.800 minutos
    fondo = m[(m["posicion"] == "central") & m["minutes"].between(400, 1800)].dropna(subset=VARIABLES).copy()
    # Kumbulla, que se asigna al grupo más cercano
    kum = m[m["jugador"] == "Marash Kumbulla"].dropna(subset=VARIABLES)
    # Recorremos las dos agrupaciones del TFM
    for k in [3, 2]:
        # k-medias con la semilla del TFM
        km = KMeans(n_clusters=k, n_init=25, random_state=42).fit(X)
        # Centros en unidades reales
        centros = pd.DataFrame(escala.inverse_transform(km.cluster_centers_), columns=NOMBRES)
        # Jugadores por grupo
        centros["jugadores"] = np.bincount(km.labels_)
        # Grupo de cada central de la muestra
        r[f"grupos{k}"] = pd.Series(km.labels_, index=base["jugador"] + " (" + base["equipo"] + ")")
        # Centros de cada grupo
        r[f"centros{k}"] = centros
        # Grupo que más anticipa
        r[f"anticipa{k}"] = int(centros["Intercepciones"].idxmax())
        # Grupo de cada central del fondo
        g = km.predict(escala.transform(fondo[VARIABLES].values))
        # Candidatos: fondo dentro del grupo que más anticipa
        r[f"candidatos{k}"] = set((fondo["jugador"] + " (" + fondo["equipo"] + ")")[g == r[f"anticipa{k}"]])
        # Grupo de Kumbulla
        r[f"kumbulla{k}"] = int(km.predict(escala.transform(kum[VARIABLES].values))[0])
    # Devolvemos el resultado
    return r


# Ejecutamos con la tabla del TFM y con la corregida
ag_a, ag_d = agrupacion(tfm), agrupacion(tfm_c)

# Comprobamos que la versión "antes" reproduce las siluetas publicadas
assert [round(ag_a["silueta"][k], 3) for k in range(2, 6)] == [0.234, 0.201, 0.210, 0.221]
# Tamaño de los grupos publicado
assert ag_a["centros3"]["jugadores"].tolist() == [25, 23, 17] and ag_a["centros2"]["jugadores"].tolist() == [31, 34]
# Número de candidatos publicado
assert len(ag_a["candidatos3"]) == 14 and len(ag_a["candidatos2"]) == 15
# Informamos
print("reproducción del TFM: correcta (siluetas, tamaño de los grupos y candidatos)")

reproducción del TFM: correcta (siluetas, tamaño de los grupos y candidatos)


In [13]:
# Silueta antes y después
print("SILUETA")
for k in range(2, 6):
    # Silueta con 6 decimales
    print(f"  k = {k}   antes {ag_a['silueta'][k]:.6f}   después {ag_d['silueta'][k]:.6f}")

# Recorremos las dos agrupaciones
for k in [3, 2]:
    # Grupos antes y después de cada jugador
    ga, gd = ag_a[f"grupos{k}"], ag_d[f"grupos{k}"]
    # Tabla cruzada de grupos
    cruce = pd.crosstab(ga, gd)
    # Cada grupo de después con el grupo de antes con el que comparte más jugadores
    pareja = cruce.idxmax(axis=0)
    # El emparejamiento tiene que ser uno a uno
    assert pareja.is_unique, "el emparejamiento de grupos no es uno a uno"
    # Grupo de después traducido a la numeración de antes
    gd_t = gd.map(pareja)
    # Jugadores que cambian de grupo
    cambian = ga[ga != gd_t]
    # Mostramos la tabla cruzada
    print(f"\nk = {k} · tabla cruzada (filas: antes, columnas: después)")
    print(cruce.to_string())
    # Mostramos quién cambia
    print(f"  jugadores que cambian de grupo: {len(cambian)}")
    for jugador in cambian.index:
        # Grupo antes y después
        print(f"    {jugador}: {ga[jugador]} -> {gd_t[jugador]}")
    # Grupo de David López del Mallorca y grupo que anticipa
    print(f"  David López (Mallorca): grupo {ga['David López (Mallorca)']} -> {gd_t['David López (Mallorca)']}"
          f" · grupo que anticipa: {ag_a[f'anticipa{k}']} -> {pareja[ag_d[f'anticipa{k}']]}")
    # Grupo de Kumbulla
    print(f"  Kumbulla: grupo {ag_a[f'kumbulla{k}']} -> {pareja[ag_d[f'kumbulla{k}']]}")
    # Centros antes
    print("  centros antes:")
    print(ag_a[f"centros{k}"].round(4).to_string())
    # Centros después, con la numeración de antes
    print("  centros después (numeración de antes):")
    print(ag_d[f"centros{k}"].rename(index=pareja).sort_index().round(4).to_string())
    # Candidatos antes y después
    ca, cd = ag_a[f"candidatos{k}"], ag_d[f"candidatos{k}"]
    print(f"  candidatos: {len(ca)} -> {len(cd)} · entran {sorted(cd - ca)} · salen {sorted(ca - cd)}")

SILUETA
  k = 2   antes 0.233722   después 0.241300
  k = 3   antes 0.200738   después 0.211325
  k = 4   antes 0.209875   después 0.214668
  k = 5   antes 0.220573   después 0.232715

k = 3 · tabla cruzada (filas: antes, columnas: después)
col_0   0   1   2
row_0            
0      24   1   0
1       1  22   0
2       0   0  17
  jugadores que cambian de grupo: 2
    Nobel Mendy (Rayo Vallecano): 1 -> 0
    Kike Salas (Sevilla): 0 -> 1
  David López (Mallorca): grupo 0 -> 0 · grupo que anticipa: 1 -> 1
  Kumbulla: grupo 0 -> 0
  centros antes:
   Intercepciones  Entradas  Despejes  Duelos aéreos  jugadores
0          0.8833    1.3901    6.5396         4.2661         25
1          1.3171    1.5664    5.7837         3.1609         23
2          0.9477    2.1004    4.2162         2.5202         17
  centros después (numeración de antes):
   Intercepciones  Entradas  Despejes  Duelos aéreos  jugadores
0          0.8639    1.3291    6.5278         4.1204         25
1          1.3381    1.6